In [76]:
# auto-reload changes to imported modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [77]:
import compile_tables as ct
import compile_file as cf
import parser_dat as p
import pandas as pd
from missing_logk import MissingSolutionSpecies

In [78]:
db_list = p.phreeqc_database_list('../databases/test_data')
print(db_list)
mst = ct.compile_master_solution_table(db_list)
sp = ct.compile_solution_species_table(db_list)
phases = ct.compile_phase_table(db_list)

['../databases/test_data/llnl.dat', '../databases/test_data/phreeqc.dat', '../databases/test_data/pitzer.dat', '../databases/test_data/minteq.v4.dat']


In [79]:
rank = {
    '#llnl.dat': 1,
    '#minteq.v4.dat': 2,
    '#phreeqc.dat': 3,
    '#Tipping_Hurley.dat': 4,
}

In [80]:
mst['priority'] = mst['source'].apply(lambda x: rank[x] if x in rank else 5)

In [81]:
# define a primary master speices by the absence of a ( or ) in the element name
mst['primary_master_species'] = mst['element'].apply(lambda x: True if '(' not in x and ')' not in x else False)
mst['secondary_master_species'] = mst['element'].apply(lambda x: True if '(' in x or ')' in x else False)
mst[mst['secondary_master_species'] == True]
((mst['primary_master_species'].astype(int) + mst['secondary_master_species'].astype(int)) == 1).all()

True

In [82]:
mst.head()

,element,species,alk,gfw_formula,element_gfw,source,priority,primary_master_species,secondary_master_species
0,Acetate,HAcetate,0.0,Acetate,59.,#llnl.dat,1,True,False
1,Ag,Ag+,0.0,Ag,107.8682,#llnl.dat,1,True,False
2,Ag(+1),Ag+,0.0,Ag,None,#llnl.dat,1,False,True
3,Ag(+2),Ag+2,0.0,Ag,None,#llnl.dat,1,False,True
4,Al,Al+3,0.0,Al,26.9815,#llnl.dat,1,True,False


In [83]:
mst.sort_values(['priority'], inplace=True)
mst.drop_duplicates(subset=['element'], keep='first', inplace=True)

In [96]:
sp[sp['equation'].str.contains('Pm+2', regex=False)]

,equation,log_k,delta_h,gamma,d_w,v_m,add_logk,no_check,source
144,Pm+3 + 0.5000 H2O = Pm+2 +H+ +0.2500 O2,-65.2754,"(0,)",None,None,None,None,None,llnl.dat


In [85]:
mst['source'].unique()

array(['#llnl.dat', '#minteq.v4.dat', '#phreeqc.dat'], dtype=object)

In [86]:
llnl = mst[mst['source'] == '#llnl.dat']
minteq = mst[mst['source'] == '#minteq.v4.dat']

In [87]:
sp.head()

,equation,log_k,delta_h,gamma,d_w,v_m,add_logk,no_check,source
0,HAcetate = HAcetate,0.0,"(0, kJ/mol)",None,None,None,None,None,llnl.dat
1,Ag+ = Ag+,0.0,"(0, kJ/mol)",None,None,None,None,None,llnl.dat
2,Al+3 = Al+3,0.0,"(0, kJ/mol)",None,None,None,None,None,llnl.dat
3,Am+3 = Am+3,0.0,"(0, kJ/mol)",None,None,None,None,None,llnl.dat
4,Ar = Ar,0.0,"(0, kJ/mol)",None,None,None,None,None,llnl.dat


In [88]:
phases.head()

,phase_name,dissolution_reaction,log_k,delta_h,analytic,v_m,t_c,p_c,omega,source
0,(UO2)2As2O7,(UO2)2As2O7 +2H+ +H2O = + 2H2AsO4- + 2UO2+2,7.7066,"(-145.281, kJ/mol)","(-1.6147e+002, -6.3487e-002, 1.0052e+004, 6.23...",None,None,None,None,llnl.dat
1,(UO2)2Cl3,(UO2)2Cl3 = + UO2+ + UO2+2 + 3Cl-,12.7339,"(-140.866, kJ/mol)","(-2.3895e+002, -9.2925e-002, 1.1722e+004, 9.69...",None,None,None,None,llnl.dat
2,(UO2)2P2O7,(UO2)2P2O7 +H2O = + 2HPO4-2 + 2UO2+2,-14.6827,"(-103.726, kJ/mol)","(-3.4581e+002, -1.3987e-001, 1.0703e+004, 1.36...",None,None,None,None,llnl.dat
3,(UO2)3(AsO4)2,(UO2)3(AsO4)2 +4H+ = + 2H2AsO4- + 3UO2+2,9.3177,"(-186.72, kJ/mol)","(-1.9693e+002, -7.3236e-002, 1.2936e+004, 7.46...",None,None,None,None,llnl.dat
4,(UO2)3(PO4)2,(UO2)3(PO4)2 +2H+ = + 2HPO4-2 + 3UO2+2,-14.0241,"(-149.864, kJ/mol)","(-3.6664e+002, -1.4347e-001, 1.3486e+004, 1.41...",None,None,None,None,llnl.dat
